# DataOracle — LLM Insights Predictor
**Phase 2 Milestone | Sheshikala | IIT Indore AI & DS**

All 5 days in one notebook:
- Day 31: model_registry — load all LLMs
- Day 32: benchmark_runner — DS tasks
- Day 33: tot_dspy_pipeline — ToT reasoning
- Day 34: pydantic_schemas — validate outputs
- Day 35: ml_report_generator — Milestone 6 report


In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
print('Ready')

## Day 31 — Model Registry

In [ ]:
from model_registry import load_all_models
models = load_all_models()
print(f'Loaded: {list(models.keys())}')

In [ ]:
# Quick test
if models:
    m = list(models.values())[0]
    r = m.chat('What is a p-value? One sentence.')
    print(f'{r.model}: {r.response[:200]}')
    print(f'Latency: {r.latency_ms}ms')

## Day 32 — LLM Benchmarking

In [ ]:
from benchmark_runner import run, save
results = run(['gpt4o','claude'], ['stat_inference'])
print(f'Results: {len(results)} entries')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if results:
    df = pd.DataFrame(results)
    pivot = df.groupby('model')['score_pct'].mean().sort_values(ascending=False)
    pivot.plot(kind='bar', figsize=(7,4), color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    plt.title('LLM Average Score on DS Tasks (%)')
    plt.ylabel('Score %'); plt.ylim(0, 100)
    plt.tight_layout()
    plt.savefig('../outputs/benchmark_chart.png', dpi=150)
    plt.show()
    json_path = save(results, out_dir='../outputs')
    print(f'Saved: {json_path}')

## Day 33 — Tree-of-Thought Reasoning

In [ ]:
try:
    from tot_dspy_pipeline import run_tot
    r = run_tot('ttest')
    print(f'Best branch: {r["best_branch"]}')
    print(f'Final answer:\n{r["final_answer"]}')
except Exception as e:
    print(f'ToT skipped: {e}')

## Day 34 — Pydantic Schemas

In [ ]:
from pydantic_schemas import HypothesisTestResult, LLMBenchmarkEntry, DataOracleReport

t = HypothesisTestResult(
    test_name='Independent t-test',
    null_hypothesis='No difference in recovery time between groups',
    alternative_hypothesis='Treatment group recovers faster',
    p_value=0.03, reject_null=True,
    conclusion='Reject H0 at alpha=0.05.',
    assumptions_met=['normality', 'equal variances']
)
print('Schema valid:', t.test_name, '| p =', t.p_value, '| reject =', t.reject_null)

## Day 35 — Milestone 6 Report

In [ ]:
from ml_report_generator import build_report, save_report
import glob

# Use benchmark results if available, else demo
json_files = sorted(glob.glob('../outputs/benchmark_*.json'))
path = json_files[-1] if json_files else None
print(f'Using: {path or "demo (no benchmark data)"}')

report = build_report(path)
md_path = save_report(report, out_dir='../outputs')
print(f'\nReport: {md_path}')

In [ ]:
from IPython.display import Markdown
Markdown(report.to_markdown())

## Milestone 6 Checklist

- [ ] Day 31: models loaded
- [ ] Day 32: benchmark ran, chart saved
- [ ] Day 33: ToT reasoning ran
- [ ] Day 34: Pydantic schema validated
- [ ] Day 35: report saved to outputs/
- [ ] All cells run with output saved
- [ ] Committed: `day10(M6): DataOracle capstone complete`
